# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import pandas as pd
import numpy as np

dataset = pd.read_csv('work/outputs/dataset.csv')

feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]

print(f"Rows: {len(dataset):,}   Clients: {dataset['client_hash_id'].nunique()}")
print(f"Declining: {dataset['is_declining_label'].sum():,} ({dataset['is_declining_label'].mean():.1%})")

desc = dataset[feature_cols].describe(percentiles=[0.5, 0.9, 0.99]).T
desc['skew'] = dataset[feature_cols].skew().round(2)
print("\n=== FEATURE DISTRIBUTIONS ===")
print(desc.round(2))

print("\n=== HEAVY-TAIL CHECK: mean vs median (traffic-count columns) ===")
traffic_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d',
                 'engaged_sessions_90d', 'organic_sessions_90d',
                 'impressions_last30', 'impressions_first60']
tail_check = pd.DataFrame({
    'mean': dataset[traffic_cols].mean(),
    'median': dataset[traffic_cols].median(),
    'p99': dataset[traffic_cols].quantile(0.99),
})
tail_check['mean/median'] = (tail_check['mean'] / tail_check['median'].replace(0, np.nan)).round(1)
print(tail_check.round(1))
print("\nA mean/median ratio well above 1 flags a heavy tail: a few very large pages")
print("dominate the average. Rule of thumb used below: ratio > ~2x -> treat as heavy-tailed")
print("and prefer bucket/quartile comparisons over raw Pearson correlation or plain means.")

print("\n=== BOUNDED / DIFFERENTLY-SHAPED COLUMNS ===")
print(dataset[['ctr_90d', 'avg_position_90d', 'momentum_pct', 'active_days_90d']].describe().round(2))
print("\nctr_90d is a %% (per the data contract, already x100 -- read as a percent, not a")
print("fraction). avg_position_90d is a search rank (lower is better, floored at 1.0 -- no")
print("zeros to worry about here, unlike the starter CSV's avg_position). momentum_pct is")
print("capped at the 99th percentile per the data contract, so still expect a long positive")
print("tail even after capping. active_days_90d is bounded by the feature window length.")


## 1. Distributions

**Traffic-count columns are heavy-tailed**, as expected for web analytics data: the
mean/median ratios printed above are well above 1 for `impressions_90d`, `clicks_90d`,
`sessions_90d`, `pageviews_90d`, `engaged_sessions_90d`, `organic_sessions_90d`,
`impressions_last30` and `impressions_first60` -- a small number of high-traffic pages pull
the mean far above the typical (median) page. Per `skills/auditing-signals/SKILL.md`, this
rules out plain Pearson correlation or raw mean comparisons on these columns -- section 2
below uses bucket/quartile comparisons instead, which is the safe default for this shape.

**`ctr_90d`** is a small bounded percentage. **`avg_position_90d`** is a search rank (lower
is better, floored at 1.0 per the ML-04 data contract -- unlike the starter CSV, there's no
`0 = no data` sentinel to filter out here). **`momentum_pct`** is capped at the 99th
percentile (~12,700% per the data contract) but still has a long right tail even after
capping, so it gets a sign-based split (negative vs. non-negative) below rather than a
magnitude comparison. **`active_days_90d`** is naturally bounded by the length of the
90-day feature window.

*(Fill in the actual mean/median ratios and skew values from the printed table above once
you run this against your real `work/outputs/dataset.csv` -- the columns named here are the
ones expected to show the heaviest tails, but confirm it.)*


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
SAMPLE_FLOOR = 50          # per skills/auditing-signals/SKILL.md: no verdict below this
MEANINGFUL_PTS = 5.0       # minimum swing (percentage points) to call a result real

def bucket_decline_rates(df, bucket_series, label_col='is_declining_label', floor=SAMPLE_FLOOR):
    """Weighted decline rate per bucket: declining rows / total rows, not a mean of rates."""
    tmp = pd.DataFrame({'bucket': bucket_series, 'label': df[label_col].values})
    tmp = tmp.dropna(subset=['bucket'])
    g = tmp.groupby('bucket', observed=True)['label'].agg(n='count', declining='sum')
    g['decline_rate'] = g['declining'] / g['n']
    g['below_floor'] = g['n'] < floor
    return g

def verdict_from_buckets(rates, floor=SAMPLE_FLOOR, meaningful_pts=MEANINGFUL_PTS, expect_increasing=True):
    """Buckets assumed ordered low->high. CONFIRMED/OPPOSITE need a monotonic, >=meaningful_pts swing."""
    if (rates['n'] < floor).any():
        return f"INSUFFICIENT DATA (a bucket has fewer than {floor} rows)"
    vals = rates['decline_rate'].values * 100
    swing = vals.max() - vals.min()
    if swing < meaningful_pts:
        return 'FALSE'
    diffs = np.diff(vals)
    increasing = (diffs >= -1e-9).all()
    decreasing = (diffs <= 1e-9).all()
    if expect_increasing:
        return 'CONFIRMED' if increasing else ('OPPOSITE' if decreasing else 'MIXED')
    else:
        return 'CONFIRMED' if decreasing else ('OPPOSITE' if increasing else 'MIXED')

def verdict_from_two_rates(rate_baseline, n_baseline, rate_test, n_test,
                            floor=SAMPLE_FLOOR, meaningful_pts=MEANINGFUL_PTS):
    """rate_test is the bucket predicted to decline MORE than rate_baseline."""
    if n_baseline < floor or n_test < floor:
        return f"INSUFFICIENT DATA (a bucket has fewer than {floor} rows)"
    diff = (rate_test - rate_baseline) * 100
    if abs(diff) < meaningful_pts:
        return 'FALSE'
    return 'CONFIRMED' if diff > 0 else 'OPPOSITE'

print("="*72)
print("TEST 1 -- Claim: pages ranking worse (higher avg_position_90d) are more likely to decline")
print("="*72)
pos_bucket = pd.qcut(dataset['avg_position_90d'], 4,
                      labels=['Q1 (best rank)', 'Q2', 'Q3', 'Q4 (worst rank)'], duplicates='drop')
test1_rates = bucket_decline_rates(dataset, pos_bucket)
print(test1_rates.assign(decline_rate_pct=(test1_rates['decline_rate']*100).round(1)))
test1_verdict = verdict_from_buckets(test1_rates, expect_increasing=True)
print(f"\nVERDICT: {test1_verdict}")

print("\n" + "="*72)
print("TEST 2 -- Claim: pages already trending down within the feature window")
print("(negative momentum_pct) are more likely to carry the declining label")
print("="*72)
has_mom = dataset[dataset['has_momentum'] == 1].copy()
print(f"Rows with has_momentum=1 (usable baseline): {len(has_mom):,} of {len(dataset):,}")
mom_bucket = pd.Series(np.where(has_mom['momentum_pct'] < 0, 'negative momentum', 'flat/positive momentum'),
                        index=has_mom.index)
test2_rates = bucket_decline_rates(has_mom, mom_bucket)
print(test2_rates.assign(decline_rate_pct=(test2_rates['decline_rate']*100).round(1)))
baseline = test2_rates.loc['flat/positive momentum']
neg = test2_rates.loc['negative momentum']
test2_verdict = verdict_from_two_rates(baseline['decline_rate'], baseline['n'], neg['decline_rate'], neg['n'])
print(f"\nVERDICT: {test2_verdict}")

print("\n" + "="*72)
print("TEST 3 -- Claim: less consistent search visibility (fewer active_days_90d)")
print("is associated with a higher decline rate")
print("="*72)
days_bucket = pd.qcut(dataset['active_days_90d'], 4,
                       labels=['Q1 (least consistent)', 'Q2', 'Q3', 'Q4 (most consistent)'], duplicates='drop')
test3_rates = bucket_decline_rates(dataset, days_bucket)
print(test3_rates.assign(decline_rate_pct=(test3_rates['decline_rate']*100).round(1)))
test3_verdict = verdict_from_buckets(test3_rates, expect_increasing=False)
print(f"\nVERDICT: {test3_verdict}")


## 2. Signal test #1 / #2 / #3 (verdict each)

Three claims, tested identically: bucket by the signal (quartiles for the two continuous
signals, a sign split for momentum since 0 is the natural pivot), require every bucket to
clear `SAMPLE_FLOOR = 50` rows before trusting it, compare the **decline rate** (declining
rows / total rows in the bucket -- a weighted rate, never an average of per-row values) across
buckets, and only call it CONFIRMED/OPPOSITE if the swing is at least `MEANINGFUL_PTS = 5`
percentage points; otherwise it's FALSE ("no real signal at this size") or MIXED
(non-monotonic across more than two buckets).

- **Test 1 -- rank position.** Claim: pages ranking worse (higher `avg_position_90d`) are
  more likely to decline. Verdict printed above.
- **Test 2 -- in-window momentum.** Claim: pages already trending down within the 90-day
  feature window (negative `momentum_pct`) are more likely to carry the declining label.
  Restricted to `has_momentum == 1` rows -- pages with `has_momentum == 0` had zero baseline
  impressions in Jan-Feb, so `momentum_pct` isn't a real trend for them (see the flag test in
  section 3). Verdict printed above.
- **Test 3 -- visibility consistency.** Claim: pages with fewer `active_days_90d` (less
  consistent search visibility within the window) decline more. Verdict printed above.

None of these three columns is on the ML-04 excluded/leak list -- all three are built only
from the 90-day feature window, before the label window starts. A CONFIRMED verdict here is
therefore a legitimate candidate signal for the Week 5 model, not leakage.

*(State the three actual verdicts here once run, e.g. "Test 1: CONFIRMED, Test 2: CONFIRMED,
Test 3: FALSE" -- and one sentence on which one surprised you.)*


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
print("="*72)
print("FLAG TEST -- has_ga4_data")
print("="*72)
print("The flag's assumption: rows with has_ga4_data=0 have sessions_90d, pageviews_90d,")
print("engaged_sessions_90d and organic_sessions_90d zero-FILLED because the CLIENT has no")
print("GA4 integration -- not because the page truly gets zero engagement. If that's right,")
print("the has_ga4_data split should mostly reflect client instrumentation, not content")
print("quality -- so it should NOT show a large, consistent difference in decline rate on")
print("its own.")
print()

ga4_bucket = pd.Series(np.where(dataset['has_ga4_data'] == 1, 'has_ga4_data=1', 'has_ga4_data=0'),
                        index=dataset.index)
flag_rates = bucket_decline_rates(dataset, ga4_bucket)
print(flag_rates.assign(decline_rate_pct=(flag_rates['decline_rate']*100).round(1)))

baseline = flag_rates.loc['has_ga4_data=1']
no_ga4 = flag_rates.loc['has_ga4_data=0']
flag_verdict = verdict_from_two_rates(baseline['decline_rate'], baseline['n'], no_ga4['decline_rate'], no_ga4['n'])
print(f"\nVERDICT: {flag_verdict}")

print("\nReading this verdict:")
print("  FALSE (no meaningful swing)  -> the flag's assumption holds: missing GA4 is just")
print("                                  missing instrumentation, not a real content-quality")
print("                                  difference. Safe to keep has_ga4_data as a structural")
print("                                  flag rather than treat it as a signal.")
print("  CONFIRMED / OPPOSITE         -> has_ga4_data is picking up something real -- worth a")
print("                                  client-level check before trusting it, since it could")
print("                                  be a client-level confound rather than a genuine")
print("                                  content-level pattern.")


## 3. The flag-linked test

`has_ga4_data` is one of the two structural flags defined in this dataset's own feature set
(ML-04 data contract) -- it exists because GA4 session/pageview columns are zero-filled per
**client**, not per page, when a client has no GA4 integration. The implicit assumption
behind treating it as a *structural* flag (rather than a real predictive signal) is that it
reflects **measurement coverage**, not **content quality** -- so on its own it shouldn't
strongly predict decline.

Verdict printed above, with a reading guide for what each outcome implies for whether
`has_ga4_data` is safe to leave out of the Week 5 model as "just a flag," or whether it
deserves a closer, client-level look first (a client confound is a leakage-adjacent risk:
the model could end up learning "which client this is" instead of "how this content is
doing").

*(State the actual verdict here once run, and if it's CONFIRMED/OPPOSITE, name which clients
are driving it.)*


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
print("="*72)
print("SIGNAL AUDIT SUMMARY")
print("="*72)
summary = pd.DataFrame([
    {'test': 'avg_position_90d -> decline (Test 1)', 'verdict': test1_verdict},
    {'test': 'momentum_pct, in-window trend -> decline (Test 2)', 'verdict': test2_verdict},
    {'test': 'active_days_90d, consistency -> decline (Test 3)', 'verdict': test3_verdict},
    {'test': 'has_ga4_data flag -> decline (flag test)', 'verdict': flag_verdict},
])
print(summary.to_string(index=False))


## 4. What this means in practice

Read the summary table above before finalizing this section. As a guide:

- Any **CONFIRMED** signal (position, momentum, or consistency) is a legitimate, non-leaky
  candidate feature for the Week 5 model -- each is built only from the 90-day feature window
  and none is on the ML-04 excluded list.
- A **FALSE** verdict is still a useful finding: it tells the content team not to over-weight
  that signal in manual review, even when it "feels" intuitively true.
- If the `has_ga4_data` flag test comes back CONFIRMED or OPPOSITE rather than FALSE, it
  shouldn't be used as a raw feature without a client-level breakdown first -- that would mean
  the flag is partly standing in for *which client* a page belongs to, not the page's own
  behavior.

> **Replace this paragraph with 2-3 sentences naming your actual verdicts** once you've run
> the cells above against your real `work/outputs/dataset.csv`, e.g.: *"Of the three signal
> tests, avg_position_90d and active_days_90d confirmed the expected relationship; momentum_pct
> did not swing the decline rate by a meaningful margin. The has_ga4_data flag showed no
> meaningful swing, supporting the assumption that it's a coverage flag, not a real signal --
> content teams should keep it out of manual scoring."*


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.